# Example 01.01 — set up the Business Case

Creates a deterministic, clearly marked Business Case for the authenticated user.
Re-running the notebook reuses the same BC and never deletes governed history.


## Connect

`MLAppClient.connect()` uses `ML_APP_ACCESS_TOKEN` when configured. Otherwise it
asks for a login and password without storing the password in the notebook.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [ ]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


## 1. Check whether the Business Case exists


In [ ]:
try:
    business_case = client.business_case_by_name(BUSINESS_CASE_NAME)
    created = False
    print(f"FOUND: {business_case['name']}")
except ResourceNotFoundError:
    business_case = None
    print("NOT FOUND: the Business Case will be created in the next cell")


## 2. Create it only when it is missing


In [ ]:
if business_case is None:
    business_case = client.create_business_case(
        name=BUSINESS_CASE_NAME,
        description="Deterministic Example 01: training, batch scoring, monitoring, and online serving.",
        problem_type="regression",
        status="draft",
        primary_metric="MAPE",
        target_column="sale_price_pln",
        business_goal="Demonstrate the complete ML lifecycle through the public API and Python client.",
        success_criteria="Process every declared row and retain reproducible lineage for every result.",
    )
    created = True
if business_case.get("access_role") not in {None, "owner"}:
    raise RuntimeError("The named example BC exists but is not owned by this account; choose another BUSINESS_CASE_NAME")
print(f"{'CREATED' if created else 'FOUND'}: {business_case['name']} ({business_case['id']})")
